# What does PCA per family cost, and why is it not the default?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [10]:
import json
import time

import polars as pl
from IPython.display import Markdown
from sklearn.decomposition import PCA

In [2]:
def get_raw_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

raw_dir = get_raw_dir()
df = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
v_cols = [c for c in df.columns if c.startswith("V")]

# Rebuild the subgroups and representatives
def get_references_dir():
    import os
    from pathlib import Path
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "references").exists():
            return current / "references"
        current = current.parent
    return Path("../../references")

references_dir = get_references_dir()
with open(references_dir / "column-groups-v.json", "r") as f:
    col_groups_json = json.load(f)

subgroups = []
assigned_cols = set()
for block in col_groups_json['blocks']:
    for group in block['groups']:
        subgroups.append(group)
        assigned_cols.update(group)

unassigned = set(v_cols) - assigned_cols
for u in unassigned:
    subgroups.append([u])

representatives = []
for group in subgroups:
    if len(group) == 1:
        representatives.append(group[0])
    else:
        uniques = [(c, df[c].n_unique()) for c in group]
        best_col = max(uniques, key=lambda x: x[1])[0]
        representatives.append(best_col)


Instead of picking a representative column out of a correlated group, PCA creates a single component that explains the variance across the entire group.

For 92 of 93 groups, a single component reproduces over 80% of the group's variance. This confirms the validity of the structural grouping.

In [4]:
df_pd = df[v_cols].to_pandas()
df_imputed = df_pd.fillna(df_pd.median())

start_time = time.time()
explained_variances = []
pca_exceptions = []
multi_groups = 0
holds_80 = 0

for group in subgroups:
    if len(group) == 1:
        continue
    
    multi_groups += 1
    pca = PCA(n_components=1)
    pca.fit(df_imputed[group])
    var = pca.explained_variance_ratio_[0]
    explained_variances.append(var)
    
    if var > 0.8:
        holds_80 += 1
    else:
        pca_exceptions.append({
            "Group": "+".join(group) if len(group) <= 4 else f"{group[0]}+{group[1]}+{group[2]}…+{group[-1]}",
            "Columns": len(group),
            "Explained variance": var
        })

runtime = time.time() - start_time


In [13]:
Markdown(explained_variances[:10].to_pandas().to_markdown(index=False))

AttributeError: 'list' object has no attribute 'to_pandas'

In [7]:
pca_exceptions

[{'Group': 'V108+V109+V110+V114',
  'Columns': 4,
  'Explained variance': np.float64(0.7337270572632483)}]